In [ ]:
import json

def get_secret(secret_name, default_value=None):
    try:
        return dbutils.secrets.get(scope="mtg-pipeline", key=secret_name)
    except Exception:
        if default_value is not None:
            return default_value
        raise

S3_BUCKET = get_secret("s3_bucket")
S3_STAGE_PREFIX = get_secret("s3_stage_prefix", "stage")
S3_BASE_PATH = f"s3://{S3_BUCKET}/{S3_STAGE_PREFIX}"

# Diretorio com escrita transacional incompleta (tem _started_<txn> mas nao
# tem _committed_<txn>/_SUCCESS - sobra de um job cancelado no meio da
# escrita). Bronze falha com UNABLE_TO_INFER_SCHEMA ao ler porque o leitor
# ignora os part-files orfaos sem transacao commitada. Limpeza manual, nao
# faz parte do codigo do pipeline (ver feedback_no_inpipe_drops). Usa mv (nao
# rm) pra ficar reversivel - so tira do caminho que save_to_parquet confere.
corrupt_path = f"{S3_BASE_PATH}/rulings/2026_09_15_rulings.parquet"
quarantine_path = f"{S3_BASE_PATH}/_quarantine/rulings/2026_09_15_rulings.parquet"

before = [f.path for f in dbutils.fs.ls(corrupt_path)]
dbutils.fs.mv(corrupt_path, quarantine_path, recurse=True)

still_there = True
try:
    dbutils.fs.ls(corrupt_path)
except Exception:
    still_there = False

moved_entries = [f.path for f in dbutils.fs.ls(quarantine_path)]

dbutils.notebook.exit(json.dumps({
    "moved_from": corrupt_path,
    "moved_to": quarantine_path,
    "had_entries": before,
    "still_exists_at_original_path": still_there,
    "entries_at_quarantine": moved_entries,
}))